## 1: Setup

In [10]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

%load_ext autoreload
%autoreload 2
from src.evaluation import evaluate

train = pd.read_csv("../data/processed/train.csv")
val = pd.read_csv("../data/processed/val.csv")
movies = pd.read_csv("../data/ml-latest-small/movies.csv")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 2: The Matrix

In [11]:
user_ids = np.sort(train["userId"].unique())
item_ids = np.sort(train["movieId"].unique())
user_index = {u: i for i, u in enumerate(user_ids)}
item_index = {m: i for i, m in enumerate(item_ids)}

rows = train["userId"].map(user_index)
cols = train["movieId"].map(item_index)
R = csr_matrix((train["rating"], (rows, cols)), shape=(len(user_ids), len(item_ids)))

print("Matrix shape (users x movies):", R.shape)
print("Ratings stored:", R.nnz)

Matrix shape (users x movies): (602, 3637)
Ratings stored: 63357


## 3: a reusable recommender builder


In [12]:
def make_recommender(scores):
    def recommend(user_id, seen, k):
        u = user_index[user_id]
        user_scores = scores[u].copy()
        seen_idx = [item_index[m] for m in seen if m in item_index]
        user_scores[seen_idx] = -np.inf
        top = np.argpartition(-user_scores, k)[:k]
        top = top[np.argsort(-user_scores[top])]
        return item_ids[top].tolist()
    return recommend

## 4: SVD scores at the chosen setting

In [13]:
svd = TruncatedSVD(n_components=15, random_state=42)
scores_svd = svd.fit_transform(R) @ svd.components_

## 5: content similarity from genres

In [14]:
movies_idx = movies.set_index("movieId").loc[item_ids]

tfidf = TfidfVectorizer(tokenizer=lambda s: s.split("|"), token_pattern=None, lowercase=False)
F = tfidf.fit_transform(movies_idx["genres"])
C = cosine_similarity(F)
print(F.shape)

(3637, 20)


## 6: content scores and a blending helper

In [15]:
scores_content = R @ C

def row_normalise(M):
    mn = M.min(axis=1, keepdims=True)
    mx = M.max(axis=1, keepdims=True)
    return (M - mn) / (mx - mn + 1e-9)

## 7: tune the blend on validation

In [16]:
results = []
for alpha in [0, 0.5, 0.7, 0.8, 0.9, 1.0]:
    hybrid = alpha * row_normalise(scores_svd) + (1 - alpha) * row_normalise(scores_content)
    r = evaluate(make_recommender(hybrid), history=train, target=val, k=10)
    r["alpha"] = alpha
    r["coverage"] = r["n_unique_items_recommended"] / len(item_ids)
    results.append(r)

table = pd.DataFrame(results).set_index("alpha")[["precision@k", "recall@k", "ndcg@k", "coverage"]].round(4)
print(table.to_string())

       precision@k  recall@k  ndcg@k  coverage
alpha                                         
0.0         0.0059    0.0063  0.0080    0.3049
0.5         0.0687    0.0931  0.1014    0.1694
0.7         0.0781    0.1082  0.1134    0.1397
0.8         0.0794    0.1047  0.1149    0.1317
0.9         0.0820    0.1088  0.1168    0.1254
1.0         0.0816    0.1109  0.1166    0.1202


#### Findings: content-based model and hybrid (validation set, k=10)

A genre-only model (TF-IDF over genre labels) scores NDCG@10 0.0080, barely above random (0.0034) and far below most-popular (0.0793), because about 20 broad genres cannot capture individual taste. Its coverage is high (30.5%) but that variety is not relevant to users.

Blending genre scores with SVD scores did not help: the best blend (alpha 0.9) has NDCG@10 0.1168 vs 0.1166 for pure SVD, a difference well within noise, and accuracy falls steadily as content weight increases. I kept pure SVD (15 components) as the final model.

Content features would matter for new movies with no ratings, which this ratings-based model cannot score. Left as future work.